In [0]:
%pip install rapidfuzz
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.1 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from rapidfuzz import fuzz

In [0]:
#Pulling in all the needed tables - BOM, warranty, sku_catalog and cleaned listings data
CATALOG = "echochain"
 
bom_raw       = spark.table(f"{CATALOG}.bronze.raw_bom")
listings_clean = spark.table(f"{CATALOG}.bronze.cleaned_listings")  #already has product_signature
sku_catalog   = spark.table(f"{CATALOG}.bronze.raw_sku_catalog")
warranty_raw  = spark.table(f"{CATALOG}.bronze.raw_warranty_claims")

In [0]:
#creating bom_id column, to uniquely identify each record
#bom_id = parent_sku + component_sku
bom_silver = bom_raw.withColumn("bom_id", F.concat_ws("_", F.col("parent_sku"), F.col("component_sku")))

In [0]:
bom_silver.head(5)

[Row(parent_sku='LAT5420-I5-16-256', parent_desc='Dell Latitude 5420 (i5-1145G7/16GB/256GB)', component_sku='MB-LAT5420-REV3', component_desc='Motherboard Rev3', component_type='Motherboard', qty_per_unit=1, unit_cost_usd=142.5, supplier='Wistron', warranty_months=12, mtbf_hours=18000, bom_id='LAT5420-I5-16-256_MB-LAT5420-REV3'),
 Row(parent_sku='LAT5420-I5-16-256', parent_desc='Dell Latitude 5420 (i5-1145G7/16GB/256GB)', component_sku='DISP-14-FHD-IPS-D1', component_desc='14in FHD IPS Display Panel', component_type='Display', qty_per_unit=1, unit_cost_usd=68.0, supplier='LG Display', warranty_months=12, mtbf_hours=45000, bom_id='LAT5420-I5-16-256_DISP-14-FHD-IPS-D1'),
 Row(parent_sku='LAT5420-I5-16-256', parent_desc='Dell Latitude 5420 (i5-1145G7/16GB/256GB)', component_sku='BATT-4CELL-68WH-D1', component_desc='4-Cell 68Wh Battery', component_type='Battery', qty_per_unit=1, unit_cost_usd=31.2, supplier='LG Chem', warranty_months=12, mtbf_hours=25000, bom_id='LAT5420-I5-16-256_BATT-4CE

In [0]:
# sanity check: bom_id should be unique — one row per (parent_sku, component_sku) pair
dupe_count = (bom_silver.groupBy("bom_id").count().filter(F.col("count") > 1).count())
assert dupe_count == 0, f"Found {dupe_count} duplicate BOM ids — check for repeated component rows"

In [0]:
#FUZZY MATCHING
#Match cleaned_listing: product_signature with sku_catalog:parent_sku

#We cannot match these two directly product_signature is more verbose and explanatory than parent_sku
#Eg: dell latitude 5420 16gb 256gb(product_signature) vs LAT5420-I5-16-256(parent_sku) --Dell Latitude 5420(product_name)
#Hence, we will create a sku_signature, where we combine the product name(brand+model), with ram,ssd extracted from parent_sku

#DIM_product — keep the full sku_catalog as-is, plus sku_signature
ram_extract = F.regexp_extract("parent_sku", r"-(\d+)-(\d+)$", 1)
storage_extract = F.regexp_extract("parent_sku", r"-(\d+)-(\d+)$", 2)

dim_product_silver = sku_catalog.withColumn(
    "sku_signature",
    F.when(
        (ram_extract != "") & (storage_extract != ""),
        F.concat_ws(" ", F.lower(F.col("product_name")),
                     F.concat(ram_extract, F.lit("gb")),
                     F.concat(storage_extract, F.lit("gb")))
    ).otherwise(F.lower(F.col("product_name")))
)

In [0]:
listings_with_sig = listings_clean
# Collect the SKU signatures to a plain Python list — it's tiny (3 products),
# so we just close over it in the UDF below rather than broadcasting, (spark.sparkContext.broadcast isn't available on serverless compute anyway; # for a list this small it wouldn't have helped much even where it is available.)
sku_catalog_local = [
    (row["parent_sku"], row["sku_signature"])
    for row in dim_product_silver.select("parent_sku", "sku_signature").collect()
]

In [0]:
sku_catalog_local

[('LAT5420-I5-16-256', 'dell latitude 5420 16gb 256gb'),
 ('ELITEBOOK840G7-I7-16-512', 'hp elitebook 840 g7 16gb 512gb'),
 ('TPADT14-R7-16-512', 'lenovo thinkpad t14 gen 1 16gb 512gb')]

In [0]:
MATCH_THRESHOLD = 70  # below this, we leave the listing unmatched rather than force a bad join
 
def fuzzy_match(signature):
    if not signature:
        return (None, 0.0)
    sig_lower = signature.lower()
    best_sku, best_primary, best_secondary = None, 0.0, 0.0
    for parent_sku, sku_signature in sku_catalog_local:
        # primary: token_set_ratio — robust to missing ram/ssd tokens on either side
        primary = fuzz.token_set_ratio(sig_lower, sku_signature)
        # tiebreaker: token_set_ratio scores 100 for ANY subset match, so a sparse
        # listing signature (e.g. brand+model only, no ram/ssd) can tie 100 against
        # multiple SKU configs of the same model. fuzz.ratio (full-string similarity)
        # breaks that tie in favor of the SKU whose signature the listing actually
        # resembles most closely overall, instead of picking whichever came first.
        secondary = fuzz.ratio(sig_lower, sku_signature)
        if (primary, secondary) > (best_primary, best_secondary):
            best_sku, best_primary, best_secondary = parent_sku, primary, secondary
    if best_primary < MATCH_THRESHOLD:
        return (None, round(best_primary, 1))
    return (best_sku, round(best_primary, 1))
 
match_schema = StructType([
    StructField("matched_parent_sku", StringType(), True),
    StructField("confidence_level", DoubleType(), True),
])
 
fuzzy_match_udf = F.udf(fuzzy_match, match_schema)
 
listings_matched = (
    listings_with_sig
    .withColumn("match_result", fuzzy_match_udf(F.col("product_signature")))
    .withColumn("matched_parent_sku", F.col("match_result.matched_parent_sku"))
    .withColumn("confidence_level", F.col("match_result.confidence_level"))
    .drop("match_result")
)

In [0]:
# quick visibility into match quality before writing anything downstream
display(
    listings_matched
    .groupBy(F.col("matched_parent_sku").isNull().alias("unmatched"))
    .count()
)
 
display(
    listings_matched.select(
        "listing_id", "product_signature", "matched_parent_sku", "confidence_level"
    ).orderBy(F.col("confidence_level").asc())
)

unmatched,count
false,10


listing_id,product_signature,matched_parent_sku,confidence_level
v1|204471190833|0,hp elitebook 840 g7 16gb 512gb,ELITEBOOK840G7-I7-16-512,100.0
v1|206612348890|0,lenovo thinkpad t14 gen 1 16gb 512gb,TPADT14-R7-16-512,100.0
v1|203847561029|0,dell latitude 5420 16gb 256gb,LAT5420-I5-16-256,100.0
v1|201938847215|0,hp elitebook 840 g7,ELITEBOOK840G7-I7-16-512,100.0
v1|207743102256|0,lenovo 16gb,TPADT14-R7-16-512,100.0
v1|198822739104|0,dell latitude 5420 16gb 256gb,LAT5420-I5-16-256,100.0
v1|211029384756|0,lenovo thinkpad t14 gen 1 16gb,TPADT14-R7-16-512,100.0
v1|210456781902|0,hp elitebook 840 g7 16gb 512gb,ELITEBOOK840G7-I7-16-512,100.0
v1|209981223470|0,dell latitude 5420 256gb,LAT5420-I5-16-256,100.0
v1|212398761045|0,dell latitude 5420,LAT5420-I5-16-256,100.0


In [0]:
# Warranty claims — carried through unchanged.
# Per the data model note: don't reconcile/correct parent_sku here against any other table — it stays exactly as generated.
warranty_silver = warranty_raw

In [0]:
#Writing everything to Silver schema as Delta Tables
(bom_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.dim_bom"))
 
(listings_matched.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.fact_listings"))
 
(warranty_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.fact_warranty_claims"))
 
(dim_product_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.dim_product"))
 
print("Silver layer written: dim_bom, fact_listings, fact_warranty_claims, dim_product")

Silver layer written: dim_bom, fact_listings, fact_warranty_claims, dim_product
